# Решения: H0/H1 и p-value

**Для преподавателя.** Эталон к `lesson.ipynb` и `homework.ipynb`. Не показывать ученикам до сдачи.

## Карта эталона

**Фокус:** От наблюдаемого uplift к проверяемой гипотезе.

Сначала измеряем конверсии A и B, затем строим нулевое распределение перестановками. Каждое число сопровождаем смыслом: что именно было зафиксировано до анализа и какой вывод допустим.

Эталон разделён на исполняемые секции в том же порядке, что `lesson.ipynb` и `homework.ipynb`. После каждой секции сверяйте не только значение, но и способ вычисления.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd


def _find(name: str) -> Path:
    for p in (Path(name), Path(f'../../data/{name}'), Path(f'../data/{name}')):
        if p.exists():
            return p.resolve()
    raise FileNotFoundError(f'{name} не найден рядом с ноутбуком')


CSV_PATH = _find('startup_ab.csv')
df = pd.read_csv(CSV_PATH)
df['variant_b'] = (df['variant'] == 'B').astype(int)


## Решение 1

Выполните секцию после всех предыдущих: переменные намеренно переиспользуются, чтобы эталон воспроизводил полный аналитический pipeline.

In [ ]:
conv_a = float(df.loc[df['variant'] == 'A', 'converted'].mean())

conv_b = float(df.loc[df['variant'] == 'B', 'converted'].mean())

uplift = conv_b - conv_a

## Решение 2

Выполните секцию после всех предыдущих: переменные намеренно переиспользуются, чтобы эталон воспроизводил полный аналитический pipeline.

In [ ]:
H0 = 'H0: конверсия одинаковая, различие A/B случайно.'

H1 = 'H1: конверсия B отличается от A (двусторонняя альтернатива).'

obs_diff = uplift

## Решение 3

Выполните секцию после всех предыдущих: переменные намеренно переиспользуются, чтобы эталон воспроизводил полный аналитический pipeline.

In [ ]:
rng = np.random.default_rng(36)

n_iter = 2000

conv = df['converted'].to_numpy()

## Решение 4

Выполните секцию после всех предыдущих: переменные намеренно переиспользуются, чтобы эталон воспроизводил полный аналитический pipeline.

In [ ]:
mask_b = df['variant'].to_numpy() == 'B'

sim_diffs = []

for _ in range(n_iter):
    perm = rng.permutation(conv)
    sim_diffs.append(float(perm[mask_b].mean() - perm[~mask_b].mean()))

## Решение 5

Выполните секцию после всех предыдущих: переменные намеренно переиспользуются, чтобы эталон воспроизводил полный аналитический pipeline.

In [ ]:
sim_diffs = np.array(sim_diffs)

p_value = float((np.abs(sim_diffs) >= abs(obs_diff)).mean())

decision = (
    'Если p-value < 0.05, отклоняем H0 и считаем эффект статистически значимым. '
    'Если p-value >= 0.05, данных недостаточно, чтобы уверенно отвергнуть случайность.'
)

## Решение 6

Выполните секцию после всех предыдущих: переменные намеренно переиспользуются, чтобы эталон воспроизводил полный аналитический pipeline.

In [ ]:
PVAL_NOTE = (
    'p-value — это вероятность получить наблюдаемое или более сильное различие, '
    'если H0 верна. Это не вероятность истинности самой H0.'
)

desktop = df[df['device'] == 'desktop']

obs_desktop = float(
    desktop.loc[desktop['variant'] == 'B', 'converted'].mean()
    - desktop.loc[desktop['variant'] == 'A', 'converted'].mean()
)

## Решение 7

Выполните секцию после всех предыдущих: переменные намеренно переиспользуются, чтобы эталон воспроизводил полный аналитический pipeline.

In [ ]:
conv_d = desktop['converted'].to_numpy()

mask_bd = desktop['variant'].to_numpy() == 'B'

sim_d = []

## Решение 8

Выполните секцию после всех предыдущих: переменные намеренно переиспользуются, чтобы эталон воспроизводил полный аналитический pipeline.

In [ ]:
for _ in range(2000):
    perm = rng.permutation(conv_d)
    sim_d.append(float(perm[mask_bd].mean() - perm[~mask_bd].mean()))

## Решение 9

Выполните секцию после всех предыдущих: переменные намеренно переиспользуются, чтобы эталон воспроизводил полный аналитический pipeline.

In [ ]:
sim_d = np.array(sim_d)

p_desktop = float((np.abs(sim_d) >= abs(obs_desktop)).mean())

p_one_sided = float((sim_d >= obs_desktop).mean())

## Решение 10

Выполните секцию после всех предыдущих: переменные намеренно переиспользуются, чтобы эталон воспроизводил полный аналитический pipeline.

In [ ]:
ONE_NOTE = 'Односторонняя альтернатива уместна только если направление эффекта задано до сбора данных.'

PROTOCOL_NOTE = (
    'Без заранее заданного протокола легко подбирать гипотезу, подвыборку и число запусков под желаемый вывод. '
    'Это резко повышает риск ложной значимости.'
)

print(round(conv_a, 4), round(conv_b, 4), round(p_value, 5), round(p_desktop, 5))

## Проверка преподавателя

Запустите `Run All`. Эталон должен завершиться без исключений; итоговые числа должны совпадать при повторном запуске благодаря фиксированным seed. Текстовый вывод проверяется на согласованность с направлением uplift, p-value и границами CI.